# Track 6 example — Can you beat "the home team wins"?
### EP0418 · Free-form starter · English Premier League results

Results data, not a sensor feed: one row per match, with the final score. The
question a free-form group can build a system around is a prediction with a
decision attached: which result to expect, and what to do with that expectation
(a Fantasy Premier League captain pick, a "watch this one" alert, nothing that
involves money).

The source is football-data.org. Its free tier covers the big European leagues
with a limit of ten requests a minute and needs a key, which takes two minutes to
get: register at football-data.org, copy the token from the e-mail, paste it
below. Keep the key out of anything you submit.

The baseline every football model has to beat is embarrassingly simple: **the home
team wins.** It is right more often than most people guess.

> **First: File → Save a copy in Drive, and work in your copy.** This page opened from GitHub, and changes here are not saved. Reopen your copy from Google Drive next time.

In [ ]:
import os, json, urllib.request
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

KEY = ""                      # paste your football-data.org token here, between the quotes
COMPETITION, SEASON = "PL", 2025

matches = None
if os.path.exists("football_matches.csv"):
    matches = pd.read_csv("football_matches.csv", parse_dates=["date"])
elif KEY:
    url = f"https://api.football-data.org/v4/competitions/{COMPETITION}/matches?season={SEASON}"
    req = urllib.request.Request(url, headers={"X-Auth-Token": KEY})
    try:
        with urllib.request.urlopen(req, timeout=60) as r:
            raw = json.load(r)["matches"]
        rows = [{"date": m["utcDate"][:10], "matchday": m["matchday"], "status": m["status"],
                 "home": m["homeTeam"]["name"], "away": m["awayTeam"]["name"],
                 "home_goals": m["score"]["fullTime"]["home"], "away_goals": m["score"]["fullTime"]["away"]}
                for m in raw]
        matches = pd.DataFrame(rows).assign(date=lambda d: pd.to_datetime(d.date))
        matches.to_csv("football_matches.csv", index=False)
    except Exception as e:
        print("football-data.org did not answer:", e)
else:
    print("No key yet. Register at football-data.org, paste the token into KEY, and run again.")

if matches is not None:
    played = matches[matches.status == "FINISHED"].copy()
    print(f"{len(matches)} fixtures, {len(played)} played, {played.date.min().date()} to {played.date.max().date()}")

### The baseline: the home team wins

Score it only on matches already played, and only on the result (win, draw, loss),
not the score. A model that predicts the exact score is solving a harder problem
than the decision needs.

In [ ]:
if matches is not None and len(played):
    played["result"] = np.sign(played.home_goals - played.away_goals)       # 1 home, 0 draw, -1 away
    acc_home = (played.result == 1).mean()
    print(f"'home team wins' is right {acc_home:.1%} of the time over {len(played)} matches")
    print(f"draws {(played.result == 0).mean():.1%}, away wins {(played.result == -1).mean():.1%}")
    print(">>> Any model has to beat that accuracy on matches it has not seen.")

    by_md = played.groupby("matchday").result.apply(lambda r: (r == 1).mean())
    fig, ax = plt.subplots(figsize=(12, 3.5))
    by_md.plot(ax=ax, marker="o"); ax.axhline(acc_home, ls="--", c="grey")
    ax.set_title("share of home wins per matchday"); ax.set_ylim(0, 1); plt.tight_layout(); plt.show()

### A fairer test: predict forward, not backward

Fit anything (home advantage per team, goals scored so far, league position) on
matchdays 1 to N, predict matchday N+1, and roll forward. Score the rolling
predictions against the home-wins baseline on the same matches. That is the Lab 5
rule, same horizon and same test set, applied to football.

In [ ]:
if matches is not None and len(played) and played.matchday.nunique() >= 8:
    hits_model, hits_base, n = 0, 0, 0
    for md in sorted(played.matchday.unique())[5:]:                 # start after five matchdays of history
        past, now = played[played.matchday < md], played[played.matchday == md]
        pts = {}                                                     # points per team so far
        for _, m in past.iterrows():
            pts[m.home] = pts.get(m.home, 0) + (3 if m.result == 1 else 1 if m.result == 0 else 0)
            pts[m.away] = pts.get(m.away, 0) + (3 if m.result == -1 else 1 if m.result == 0 else 0)
        for _, m in now.iterrows():
            pred = 1 if pts.get(m.home, 0) + 3 >= pts.get(m.away, 0) else -1   # better team (plus home edge) wins
            hits_model += pred == m.result; hits_base += m.result == 1; n += 1
    print(f"rolling forward over {n} matches:  'better team wins' {hits_model / n:.1%}   "
          f"'home team wins' {hits_base / n:.1%}")

---
## Three macro questions to shape a proposal

**1. What decision follows from the prediction?**
"Arsenal will probably win" is a number. "Captain the Arsenal striker this week"
or "flag this fixture as an upset risk" is a decision with a person attached.
Nothing in this track involves betting (SLO 7.2).

**2. What does the baseline know that your model does not?**
Home advantage is real and large. A model that ignores it will lose to the
baseline, and a model that only encodes it will tie. Where is the extra
information: form, injuries, rest days, derby matches? (SLO 6.2)

**3. How will you know next season?**
Fixtures, teams and rules change. The system has to re-score itself against the
baseline each matchday and say when it has stopped working (SLO 9.1, 10.1).